In [1]:
# %%
# -*- coding: utf-8 -*-
"""
backend8_total_mes_loss_time.ipynb (single-run notebook)

사양/확정 규칙 반영:
- prod_day 1개 입력 → day/night 결과를 각각 별도 DF로 출력
- 출력은 dataframe display까지 (DB 저장 없음)
- KST(Asia/Seoul) 기준 shift window:
  day   : [D] 08:30:00 ~ 20:29:59 (inclusive)
  night : [D] 20:30:00 ~ [D+1] 08:29:59 (inclusive)
- mes_fail_wasted_time:
  - end_day = from_time 기준 날짜
  - from_time/to_time 소수초는 .5 half-up 반올림 후 HH:MI:SS 정규화
  - wasted_time은 DB 값을 신뢰
  - 시간 겹침(단일 row가 shift 경계를 가로지름) 시:
      * 손실시간(분할된 sec)이 더 큰 shift에 count 1 부여
      * tie(day=night)이면 day로 count 부여
  - 경계 규칙(예시 반영):
      * to_time == 08:30:00이면 그 1초는 day 귀속
      * from_time == 20:30:00이면 night 포함
    → 구현은 "초를 end timestamp로 카운트" 방식( (from, to] )
- vision_op_ct:
  - month = window_start 기준 이전달(YYYYMM) 사용
  - station in ('Vision1_only','Vision2_only'), remark='PD'
  - station별 행이 2개 이상이면 updated_at 최신 1개 선택
  - 재작업 시간 = (del_out_op_ct_av(V1)+del_out_op_ct_av(V2)) / 4 * MES 불량개수
- mes_fail2_wasted_time:
  - shift window에 들어오는 행들의 final_ct 합산 (NULL이면 0)
- Total = 3개 항 합을 .5 half-up 반올림
- 출력 문자열: "h시간 m분 s초" 단 0 단위 제외
- updated_at: KST 기준 now() (timezone-aware)
"""

from __future__ import annotations

import os
import math
from dataclasses import dataclass
from datetime import datetime, date, time, timedelta
from decimal import Decimal, ROUND_HALF_UP
from zoneinfo import ZoneInfo
from typing import Optional, Tuple, Dict

import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import Engine

KST = ZoneInfo("Asia/Seoul")

# =========================
# 0) 입력
# =========================
PROD_DAY = "20260130"  # YYYYMMDD (여기만 바꿔서 실행)

# =========================
# 1) DB 설정 (사양서 기본값)
#    보안상 env 우선 사용(없으면 기본값)
# =========================
DB_CONFIG = {
    "host": os.getenv("PGHOST", "100.105.75.47"),
    "port": int(os.getenv("PGPORT", "5432")),
    "dbname": os.getenv("PGDATABASE", "postgres"),
    "user": os.getenv("PGUSER", "postgres"),
    "password": os.getenv("PGPASSWORD", ""),
}

MES_SCHEMA = "d1_machine_log"
T_MES_FAIL = "mes_fail_wasted_time"
T_MES_FAIL2 = "mes_fail2_wasted_time"

VISION_SCHEMA = "e2_vision_ct"
T_VISION_OP_CT = "vision_op_ct"


def make_engine() -> Engine:
    url = (
        f"postgresql+psycopg2://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
        f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['dbname']}"
    )
    return create_engine(
        url,
        pool_size=1,
        max_overflow=0,
        pool_pre_ping=True,
        pool_recycle=1800,
    )


# =========================
# 2) 유틸: half-up 라운딩/시간 파싱/포맷
# =========================
def round_half_up_int(x) -> int:
    # float/Decimal/str 모두 처리
    d = Decimal(str(x))
    return int(d.quantize(Decimal("1"), rounding=ROUND_HALF_UP))


def parse_time_text_halfup(t: str) -> Tuple[int, int, int]:
    """
    'HH:MI:SS' 또는 'HH:MI:SS.xx' 를 half-up 반올림하여 (h,m,s) 반환.
    반올림 carry로 24:00:00 가능.
    """
    t = (t or "").strip()
    if not t:
        return (0, 0, 0)

    parts = t.split(":")
    if len(parts) != 3:
        raise ValueError(f"Invalid time format: {t}")

    hh = int(parts[0])
    mm = int(parts[1])

    sec_part = parts[2]
    if "." in sec_part:
        ss_f = Decimal(sec_part)
        ss = int(ss_f.quantize(Decimal("1"), rounding=ROUND_HALF_UP))
    else:
        ss = int(sec_part)

    # carry 처리
    if ss >= 60:
        carry_m = ss // 60
        ss = ss % 60
        mm += carry_m
    if mm >= 60:
        carry_h = mm // 60
        mm = mm % 60
        hh += carry_h

    return (hh, mm, ss)


def dt_from_end_day_and_time(end_day: str, hh: int, mm: int, ss: int) -> datetime:
    """
    end_day(YYYYMMDD) + (hh,mm,ss) → timezone-aware KST datetime.
    hh가 24 이상이면 날짜 carry.
    """
    d = datetime.strptime(end_day, "%Y%m%d").date()
    add_days = hh // 24
    hh2 = hh % 24
    d2 = d + timedelta(days=add_days)
    return datetime(d2.year, d2.month, d2.day, hh2, mm, ss, tzinfo=KST)


def format_hms_korean(total_sec: int) -> str:
    total_sec = int(total_sec)
    if total_sec <= 0:
        return "0초"
    h = total_sec // 3600
    m = (total_sec % 3600) // 60
    s = total_sec % 60
    parts = []
    if h:
        parts.append(f"{h}시간")
    if m:
        parts.append(f"{m}분")
    if s:
        parts.append(f"{s}초")
    return " ".join(parts)


def prev_month_yyyymm_by_window_start(prod_day: str) -> str:
    """
    [window 기준] 이전달:
    - prod_day의 day/night window_start는 모두 prod_day 날짜에 존재(08:30/20:30)
    - 그 날짜가 속한 달 기준 -1month
    예) 20260127 -> 202512
    """
    d = datetime.strptime(prod_day, "%Y%m%d").date()
    first = date(d.year, d.month, 1)
    prev_last = first - timedelta(days=1)
    return f"{prev_last.year:04d}{prev_last.month:02d}"


# =========================
# 3) Shift window 생성
# =========================
@dataclass(frozen=True)
class ShiftWindow:
    shift: str
    start: datetime
    end: datetime

def build_windows(prod_day: str) -> Dict[str, ShiftWindow]:
    d = datetime.strptime(prod_day, "%Y%m%d").date()
    day_start = datetime(d.year, d.month, d.day, 8, 30, 0, tzinfo=KST)
    day_end   = datetime(d.year, d.month, d.day, 20, 29, 59, tzinfo=KST)
    night_start = datetime(d.year, d.month, d.day, 20, 30, 0, tzinfo=KST)
    d2 = d + timedelta(days=1)
    night_end = datetime(d2.year, d2.month, d2.day, 8, 29, 59, tzinfo=KST)
    return {
        "day": ShiftWindow("day", day_start, day_end),
        "night": ShiftWindow("night", night_start, night_end),
    }


# =========================
# 4) mes_fail_wasted_time: (from, to] 초 end-timestamp 카운트 기반 분할
# =========================
def count_end_seconds_in_window(from_int: int, to_int: int, win_a: int, win_b: int) -> int:
    """
    end-second counting:
      seconds correspond to integer end timestamps s in (from_int, to_int]  => from_int+1 .. to_int
    window includes endpoints [win_a, win_b] (inclusive)
    """
    if to_int <= from_int:
        return 0
    lo = max(from_int + 1, win_a)
    hi = min(to_int, win_b)
    if hi < lo:
        return 0
    return hi - lo + 1


def split_row_to_day_night(
    end_day: str,
    from_time_txt: str,
    to_time_txt: str,
    wasted_time_db,
    win_day: ShiftWindow,
    win_night: ShiftWindow,
) -> Tuple[int, int, int, int, bool]:
    """
    반환:
      day_w_sec, night_w_sec, day_cnt_flag, night_cnt_flag, crossed
    - day/night 손실시간(sec)은 DB wasted_time을 기준으로 총합을 맞춤(필요 시 비율 배분)
    - count는 crossed 여부에 따라 한쪽에만 1
    """
    # 1) time 반올림 정규화
    fh, fm, fs = parse_time_text_halfup(from_time_txt)
    th, tm, ts = parse_time_text_halfup(to_time_txt)

    from_dt = dt_from_end_day_and_time(end_day, fh, fm, fs)
    to_dt   = dt_from_end_day_and_time(end_day, th, tm, ts)

    # 2) 자정 넘어감 처리: to_dt <= from_dt 이면 다음날로
    if to_dt <= from_dt:
        to_dt = to_dt + timedelta(days=1)

    # 3) 정수초(UTC epoch)로 변환
    from_int = int(from_dt.timestamp())
    to_int   = int(to_dt.timestamp())

    # 4) total seconds by spec counting: (from, to] => to - from
    total_s = max(0, to_int - from_int)

    # 5) window endpoints int
    day_a, day_b = int(win_day.start.timestamp()), int(win_day.end.timestamp())
    night_a, night_b = int(win_night.start.timestamp()), int(win_night.end.timestamp())

    # 6) split seconds (grid-based)
    day_s = count_end_seconds_in_window(from_int, to_int, day_a, day_b)
    night_s = count_end_seconds_in_window(from_int, to_int, night_a, night_b)

    # 7) DB wasted_time 신뢰(half-up int). split 합이 다르면 비율 배분으로 맞춤
    total_w = round_half_up_int(wasted_time_db)

    if total_s <= 0:
        # 이상 케이스: duration 0인데 wasted_time이 존재할 수 있음 → to_dt가 속한 shift에 몰아줌
        # (to_dt의 "end second"는 to_int)
        if day_a <= to_int <= day_b:
            return total_w, 0, 1, 0, False
        if night_a <= to_int <= night_b:
            return 0, total_w, 0, 1, False
        # 어떤 window에도 안 잡히면 버림(현실적으로는 없음)
        return 0, 0, 0, 0, False

    # split 기준: day_s, night_s 합이 total_s와 약간 다를 수 있으니 total_s 기반으로 비율 배분
    # (하지만 정상이라면 day_s + night_s == total_s)
    if day_s + night_s == 0:
        # window 밖: 버림
        return 0, 0, 0, 0, False

    # 분할 wasted_time (정수 sec, 합은 total_w)
    # total_s 대신 (day_s+night_s)로 분모를 잡아 안전하게 처리
    denom = day_s + night_s
    day_w = round_half_up_int(Decimal(total_w) * Decimal(day_s) / Decimal(denom))
    night_w = total_w - day_w

    crossed = (day_s > 0 and night_s > 0)

    # count 배정
    if not crossed:
        if day_s > 0 and night_s == 0:
            return day_w, 0, 1, 0, False
        if night_s > 0 and day_s == 0:
            return 0, night_w, 0, 1, False
        # 둘 다 >0인데 crossed 판정이 False일 수는 없음(안전)
        # tie면 day
        return day_w, night_w, 1, 0, True
    else:
        # "더 많은 손실시간" 기준, tie면 day
        if day_w > night_w:
            return day_w, night_w, 1, 0, True
        if night_w > day_w:
            return day_w, night_w, 0, 1, True
        return day_w, night_w, 1, 0, True


def compute_mes_fail_loss(engine: Engine, prod_day: str, windows: Dict[str, ShiftWindow]) -> Dict[str, Dict[str, int]]:
    """
    결과:
      {
        "day":   {"loss_sec": int, "count": int},
        "night": {"loss_sec": int, "count": int},
      }
    """
    d = datetime.strptime(prod_day, "%Y%m%d").date()
    next_day = (d + timedelta(days=1)).strftime("%Y%m%d")

    sql = text(f"""
        SELECT end_day, from_time, to_time, wasted_time
        FROM {MES_SCHEMA}.{T_MES_FAIL}
        WHERE end_day IN (:d1, :d2)
    """)

    df = pd.read_sql(sql, engine, params={"d1": prod_day, "d2": next_day})

    day_loss = day_cnt = 0
    night_loss = night_cnt = 0

    for r in df.itertuples(index=False):
        end_day = str(r.end_day)
        from_t = str(r.from_time)
        to_t = str(r.to_time)
        wasted = r.wasted_time

        day_w, night_w, day_c, night_c, _ = split_row_to_day_night(
            end_day=end_day,
            from_time_txt=from_t,
            to_time_txt=to_t,
            wasted_time_db=wasted,
            win_day=windows["day"],
            win_night=windows["night"],
        )
        day_loss += int(day_w)
        night_loss += int(night_w)
        day_cnt += int(day_c)
        night_cnt += int(night_c)

    return {
        "day": {"loss_sec": day_loss, "count": day_cnt},
        "night": {"loss_sec": night_loss, "count": night_cnt},
    }


# =========================
# 5) vision_op_ct: 이전달 + station별 최신 1개
# =========================
def fetch_del_out_op_ct_av(engine: Engine, prev_month: str) -> Dict[str, float]:
    sql = text(f"""
        SELECT DISTINCT ON (station)
               station,
               del_out_op_ct_av,
               updated_at
        FROM {VISION_SCHEMA}.{T_VISION_OP_CT}
        WHERE month = :m
          AND station IN ('Vision1_only', 'Vision2_only')
          AND remark = 'PD'
        ORDER BY station, updated_at DESC NULLS LAST
    """)
    df = pd.read_sql(sql, engine, params={"m": prev_month})
    out = {"Vision1_only": 0.0, "Vision2_only": 0.0}
    for r in df.itertuples(index=False):
        out[str(r.station)] = float(r.del_out_op_ct_av) if r.del_out_op_ct_av is not None else 0.0
    return out


def compute_rework_time_sec(del_out: Dict[str, float], mes_fail_cnt: int) -> int:
    # (V1 + V2) / 4 * count  → .5 half-up
    total = (float(del_out.get("Vision1_only", 0.0)) + float(del_out.get("Vision2_only", 0.0))) / 4.0
    return round_half_up_int(total * mes_fail_cnt)


# =========================
# 6) mes_fail2_wasted_time: shift window 내 final_ct 합
# =========================
def compute_fct_rework_time(engine: Engine, prod_day: str, shift: str) -> int:
    d = datetime.strptime(prod_day, "%Y%m%d").date()
    next_day = (d + timedelta(days=1)).strftime("%Y%m%d")

    if shift == "day":
        sql = text(f"""
            SELECT COALESCE(SUM(COALESCE(final_ct, 0)), 0) AS s
            FROM {MES_SCHEMA}.{T_MES_FAIL2}
            WHERE end_day = :d
              AND CAST(end_time AS time) BETWEEN TIME '08:30:00' AND TIME '20:29:59'
        """)
        val = pd.read_sql(sql, engine, params={"d": prod_day}).iloc[0, 0]
        return round_half_up_int(val)

    if shift == "night":
        sql = text(f"""
            SELECT COALESCE(SUM(COALESCE(final_ct, 0)), 0) AS s
            FROM {MES_SCHEMA}.{T_MES_FAIL2}
            WHERE (end_day = :d  AND CAST(end_time AS time) >= TIME '20:30:00')
               OR (end_day = :d2 AND CAST(end_time AS time) <= TIME '08:29:59')
        """)
        val = pd.read_sql(sql, engine, params={"d": prod_day, "d2": next_day}).iloc[0, 0]
        return round_half_up_int(val)

    raise ValueError("shift must be 'day' or 'night'")


# =========================
# 7) 메인 실행
# =========================
engine = make_engine()

windows = build_windows(PROD_DAY)
prev_month = prev_month_yyyymm_by_window_start(PROD_DAY)

# 3-1
mes_fail = compute_mes_fail_loss(engine, PROD_DAY, windows)

# 3-2 (이전달 del_out_op_ct_av)
del_out = fetch_del_out_op_ct_av(engine, prev_month)

# 3-3
fct_day = compute_fct_rework_time(engine, PROD_DAY, "day")
fct_night = compute_fct_rework_time(engine, PROD_DAY, "night")

now_kst = datetime.now(KST)

# day
day_cnt = mes_fail["day"]["count"]
day_loss = mes_fail["day"]["loss_sec"]
day_rework = compute_rework_time_sec(del_out, day_cnt)
day_total = round_half_up_int(Decimal(day_loss + day_rework + fct_day))
df_day = pd.DataFrame([{
    "prod_day": PROD_DAY,
    "shift_type": "day",
    "MES 불량 손실 시간(초)": int(day_loss),
    "MES 불량 제품 개수": int(day_cnt),
    "재작업 시간(초)": int(day_rework),
    "FCT 재작업 시간(초)": int(fct_day),
    "Total MES 불량 손실 시간(초)": int(day_total),
    "Total MES 불량 손실 시간": format_hms_korean(int(day_total)),
    "updated_at": now_kst,
}])

# night
night_cnt = mes_fail["night"]["count"]
night_loss = mes_fail["night"]["loss_sec"]
night_rework = compute_rework_time_sec(del_out, night_cnt)
night_total = round_half_up_int(Decimal(night_loss + night_rework + fct_night))
df_night = pd.DataFrame([{
    "prod_day": PROD_DAY,
    "shift_type": "night",
    "MES 불량 손실 시간(초)": int(night_loss),
    "MES 불량 제품 개수": int(night_cnt),
    "재작업 시간(초)": int(night_rework),
    "FCT 재작업 시간(초)": int(fct_night),
    "Total MES 불량 손실 시간(초)": int(night_total),
    "Total MES 불량 손실 시간": format_hms_korean(int(night_total)),
    "updated_at": now_kst,
}])

In [3]:
# %%
import pandas as pd
from datetime import datetime
from zoneinfo import ZoneInfo
from IPython.display import display

KST = ZoneInfo("Asia/Seoul")

def format_hms_korean(total_sec: int) -> str:
    total_sec = int(total_sec) if total_sec is not None else 0
    if total_sec <= 0:
        return "0초"
    h = total_sec // 3600
    m = (total_sec % 3600) // 60
    s = total_sec % 60
    parts = []
    if h:
        parts.append(f"{h}시간")
    if m:
        parts.append(f"{m}분")
    if s:
        parts.append(f"{s}초")
    return " ".join(parts)

def safe_int(x) -> int:
    try:
        if x is None or (isinstance(x, float) and pd.isna(x)) or pd.isna(x):
            return 0
        return int(x)
    except Exception:
        return 0

# ✅ prod_day 표시를 "YYYY-MM-DD"로 (스크린샷처럼)
prod_day_display = pd.to_datetime(PROD_DAY, format="%Y%m%d").date()
now_kst = datetime.now(KST)

# ---- day 값 정리(없으면 0) ----
_day_loss   = safe_int(day_loss)
_day_cnt    = safe_int(day_cnt)
_day_rework = safe_int(day_rework)
_fct_day    = safe_int(fct_day)
_day_total  = safe_int(day_total)

# ---- night 값 정리(없으면 0) ----
_night_loss   = safe_int(night_loss)
_night_cnt    = safe_int(night_cnt)
_night_rework = safe_int(night_rework)
_fct_night    = safe_int(fct_night)
_night_total  = safe_int(night_total)

# ✅ 컬럼명 스크린샷 그대로
COLS = [
    "prod_day",
    "shift_type",
    "MES 불량 손실시간(초)",
    "MES 불량 제품 개수",
    "재작업 시간(초)",
    "FCT 재작업 시간(초)",
    "Total MES 불량 손실 시간(초)",
    "Total MES 불량 손실 시간",
    "updated_at",
]

df_day = pd.DataFrame([{
    "prod_day": prod_day_display,
    "shift_type": "day",
    "MES 불량 손실시간(초)": _day_loss,
    "MES 불량 제품 개수": _day_cnt,
    "재작업 시간(초)": _day_rework,
    "FCT 재작업 시간(초)": _fct_day,
    "Total MES 불량 손실 시간(초)": _day_total,
    "Total MES 불량 손실 시간": format_hms_korean(_day_total),
    "updated_at": now_kst,
}])[COLS]

df_night = pd.DataFrame([{
    "prod_day": prod_day_display,
    "shift_type": "night",
    "MES 불량 손실시간(초)": _night_loss,
    "MES 불량 제품 개수": _night_cnt,
    "재작업 시간(초)": _night_rework,
    "FCT 재작업 시간(초)": _fct_night,
    "Total MES 불량 손실 시간(초)": _night_total,
    "Total MES 불량 손실 시간": format_hms_korean(_night_total),
    "updated_at": now_kst,
}])[COLS]

# ✅ day / night 따로 연속 출력 (한 셀에서 둘 다 보이게)
display(df_day)
display(df_night)

,prod_day,shift_type,MES 불량 손실시간(초),MES 불량 제품 개수,재작업 시간(초),FCT 재작업 시간(초),Total MES 불량 손실 시간(초),Total MES 불량 손실 시간,updated_at
0,2026-01-30,day,779,19,185,144,1108,18분 28초,2026-02-01 12:00:44.192768+09:00


,prod_day,shift_type,MES 불량 손실시간(초),MES 불량 제품 개수,재작업 시간(초),FCT 재작업 시간(초),Total MES 불량 손실 시간(초),Total MES 불량 손실 시간,updated_at
0,2026-01-30,night,335,10,97,186,618,10분 18초,2026-02-01 12:00:44.192768+09:00


In [4]:
# %%
from sqlalchemy import text
import pandas as pd

SAVE_SCHEMA = "i_daily_report"
T_DAY   = "h_mes_wasted_time_day_daily"
T_NIGHT = "h_mes_wasted_time_night_daily"

def ensure_schema(engine, schema: str):
    with engine.begin() as conn:
        conn.execute(text(f'CREATE SCHEMA IF NOT EXISTS "{schema}";'))

def ensure_table(engine, schema: str, table: str):
    """
    df_day/df_night 컬럼 스펙에 맞춰 테이블 생성 + prod_day UNIQUE
    """
    ddl = f"""
    CREATE TABLE IF NOT EXISTS "{schema}"."{table}" (
        prod_day date NOT NULL,
        shift_type text,
        "MES 불량 손실시간(초)" integer,
        "MES 불량 제품 개수" integer,
        "재작업 시간(초)" integer,
        "FCT 재작업 시간(초)" integer,
        "Total MES 불량 손실 시간(초)" integer,
        "Total MES 불량 손실 시간" text,
        updated_at timestamptz,
        CONSTRAINT "{table}__uq_prod_day" UNIQUE (prod_day)
    );
    """
    with engine.begin() as conn:
        conn.execute(text(ddl))

def upsert_one(engine, schema: str, table: str, row: dict):
    """
    prod_day 기준 UPSERT
    - NULL 덮어쓰기 방지: COALESCE(EXCLUDED.col, target.col)
    """
    sql = text(f"""
    INSERT INTO "{schema}"."{table}" (
        prod_day, shift_type,
        "MES 불량 손실시간(초)", "MES 불량 제품 개수",
        "재작업 시간(초)", "FCT 재작업 시간(초)",
        "Total MES 불량 손실 시간(초)", "Total MES 불량 손실 시간",
        updated_at
    ) VALUES (
        :prod_day, :shift_type,
        :mes_loss_sec, :mes_cnt,
        :rework_sec, :fct_rework_sec,
        :total_sec, :total_hms,
        :updated_at
    )
    ON CONFLICT (prod_day) DO UPDATE SET
        shift_type = COALESCE(EXCLUDED.shift_type, "{schema}"."{table}".shift_type),
        "MES 불량 손실시간(초)" = COALESCE(EXCLUDED."MES 불량 손실시간(초)", "{schema}"."{table}"."MES 불량 손실시간(초)"),
        "MES 불량 제품 개수" = COALESCE(EXCLUDED."MES 불량 제품 개수", "{schema}"."{table}"."MES 불량 제품 개수"),
        "재작업 시간(초)" = COALESCE(EXCLUDED."재작업 시간(초)", "{schema}"."{table}"."재작업 시간(초)"),
        "FCT 재작업 시간(초)" = COALESCE(EXCLUDED."FCT 재작업 시간(초)", "{schema}"."{table}"."FCT 재작업 시간(초)"),
        "Total MES 불량 손실 시간(초)" = COALESCE(EXCLUDED."Total MES 불량 손실 시간(초)", "{schema}"."{table}"."Total MES 불량 손실 시간(초)"),
        "Total MES 불량 손실 시간" = COALESCE(EXCLUDED."Total MES 불량 손실 시간", "{schema}"."{table}"."Total MES 불량 손실 시간"),
        updated_at = COALESCE(EXCLUDED.updated_at, "{schema}"."{table}".updated_at)
    ;
    """)

    params = {
        "prod_day": row["prod_day"],
        "shift_type": row["shift_type"],
        "mes_loss_sec": int(row.get("MES 불량 손실시간(초)", 0) or 0),
        "mes_cnt": int(row.get("MES 불량 제품 개수", 0) or 0),
        "rework_sec": int(row.get("재작업 시간(초)", 0) or 0),
        "fct_rework_sec": int(row.get("FCT 재작업 시간(초)", 0) or 0),
        "total_sec": int(row.get("Total MES 불량 손실 시간(초)", 0) or 0),
        "total_hms": row.get("Total MES 불량 손실 시간", None),
        "updated_at": row.get("updated_at", None),
    }

    with engine.begin() as conn:
        conn.execute(sql, params)

# ---- 실행 ----
# engine 변수가 이미 위에서 만들어져 있어야 함 (없으면 make_engine()로 생성)
ensure_schema(engine, SAVE_SCHEMA)
ensure_table(engine, SAVE_SCHEMA, T_DAY)
ensure_table(engine, SAVE_SCHEMA, T_NIGHT)

# df_day/df_night는 1행 DF 전제
day_row = df_day.iloc[0].to_dict()
night_row = df_night.iloc[0].to_dict()

upsert_one(engine, SAVE_SCHEMA, T_DAY, day_row)
upsert_one(engine, SAVE_SCHEMA, T_NIGHT, night_row)

print(f"[OK] saved day  -> {SAVE_SCHEMA}.{T_DAY} (prod_day={day_row['prod_day']})")
print(f"[OK] saved night-> {SAVE_SCHEMA}.{T_NIGHT} (prod_day={night_row['prod_day']})")

[OK] saved day  -> i_daily_report.h_mes_wasted_time_day_daily (prod_day=2026-01-30)
[OK] saved night-> i_daily_report.h_mes_wasted_time_night_daily (prod_day=2026-01-30)
